# Counterfactual-History State — does a clean-edit hidden state `h*` exist?

**Direction:** `research/directions/counterfactual-history-state.md` · `[in-frame]` · sub-Q3 (editability). **Reference sanity check** (expected to succeed ~tautologically), *not* a headline — only surprising if `h*` fails to render the teleport cleanly.

## The question
Latent editing writes `h` directly and is **not** bound by the model's observation-mediated belief update. The master notebook's **true-state swap** reference (`h_gt`) teacher-forces the *realized* post-edit observations up to the edit frame `ef` — but those contain a **teleport discontinuity** at `ef`, so the model gets only **one frame** of teleport evidence. Belief inertia then leaves a **ghost** at the old location. That is a *lower bound* on editing quality, not a ceiling.

This experiment asks the existence question directly: **does a hidden state `h*` exist that, injected at the edit frame, renders the teleport cleanly** — edited object at target, ghost-free, and persisting over a rollout?

- **If yes** (expected): the editing failure localizes to the **edit map's reachability** — the target state exists in `h`-space and carries all the information; a low-dimensional probe-injection simply cannot *reach* it. This sharpens the learn-to-edit negative result.
- **If no** (surprising): flag loudly — either injection ≠ teacher-forcing, or the model cannot even represent the counterfactual.

## Construction of `h*` (per edit sample)
Build the **counterfactual history**: the edited object is back-extrapolated from the target at `ef` along its *preserved* velocity, so it approaches the target smoothly over frames `0..ef` (no discontinuity). The other object keeps its **true** history. Render clean observations `0..ef` of this counterfactual world, teacher-force the GRU → `h*` (state after frame `ef`). `h*` and `h_gt` end at the **same target frame**; they differ only in the *history* the model saw.

## Data-source provenance
- **Model:** GRU `runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt` (only; GRU is sufficient per brief).
- **Data:** `datasets/4_fixed_refl_inview` — `edits` split, `edit_frame = ef` (uniform), `N = 64` edit samples.
- **GT reference:** `edits.clean_obs[:, ef:ef+K]` — the simulator's true post-edit observations (never a model output). From `ef` onward the counterfactual world **equals** the post-edit world, so this is the correct target for `h*`.
- **Rollout↔frame alignment (mirrors master §4):** a state that has consumed frames `0..m-1` decodes a prediction of frame `m`; rollout **step `s` ↔ sim frame `ef+s`**. `h_gt`/`h*` consume through `ef`; `h0` (unsteered) consumes through `ef-1`. Conventions are held identical across all states so the head-to-head is apples-to-apples.

PNGs → `/tmp/counterfactual_history/`. Cells tagged `# [N]`; figures numbered.

## Definitions — states compared, and every metric with its formula

**States rolled out** (all consume clean observations through the stated frame, then roll out `K=15` steps; step `s` ↔ sim frame `ef+s`):

| symbol | name | how built | role |
|---|---|---|---|
| **GT** | ground truth | `edits.clean_obs[:, ef:ef+K]` (sim, not a model output) | target / floor |
| **`h*`** | counterfactual-history state | teacher-force clean obs of the *counterfactual* world (edited object back-extrapolated to pass through target at `ef` with preserved velocity; other object true history), frames `0..ef` | existence test |
| **`h*_shared`** | shared-context variant | real clean obs for frames `0..ef-W`, counterfactual clean obs for the last `W=10` frames | frustum-robust `h*` |
| **`h_gt`** | one-frame-evidence (true-state swap) | teacher-force *realized* post-edit clean obs `0..ef` (contains the teleport discontinuity at `ef`) | lower-bound reference |
| **`h0`** | unsteered | teacher-force realized clean obs `0..ef-1` (never sees the teleport) | ghost baseline |
| **`h_readout`** | readout injection | inject target position into `h0` via the linear position-probe pseudoinverse (null-space preserved) | known-failing edit |

**Metrics** (observation intensities are in `[0,1]`; obs resolution `R=128` rays):

| metric | formula | units | better |
|---|---|---|---|
| **obs-RMSE-to-GT** (step `s`) | `sqrt(mean_rays,samples ( rollout[:,s,:] − GT[:,s,:] )²)` | intensity RMSE | ↓ (0 = matches sim) |
| **ghost ratio** (step `s`) | `mean(rollout[:,s,:][G]) / mean(h0_rollout[:,s,:][G])`, `G` = ghost rays | ratio | ↓ (0 = ghost gone, 1 = full ghost) |
| **target-fill ratio** (step `s`) | `mean(rollout[:,s,:][T]) / mean(GT[:,s,:][T])`, `T` = target rays | ratio | →1 (object present at target like GT) |
| **ghost rays `G`** | rays lit by the edited object **pre-edit** (`ef-1`) but **not** at target (`ef`) | ray set | — |
| **target rays `T`** | rays lit by the edited object **at target** (`ef`) | ray set | — |
| **‖h* − h0‖** etc. | mean over samples of the L2 distance between two flat hidden states | L2 in `h`-space | — |
| **probe-aligned fraction** | `‖P_row(A)(h*−h0)‖ / ‖h*−h0‖`, `P_row(A)` = projection onto the row space of the linear position probe `A` (the subspace a pseudoinverse position-injection can move within) | fraction ∈ [0,1] | — (low ⇒ `h*` unreachable by position injection) |

**Persistence** is read as the metric holding across `s`: obs-RMSE-to-GT staying low, ghost staying ≈0, and target-fill staying ≈1 from step 0 to step `K-1`.

**Decision rule.** `h*` renders cleanly iff its obs-RMSE-to-GT is **≪** `h_gt`'s, its ghost ≈ 0, and it persists. Then a clean-edit state **exists** in `h`-space ⇒ the editing failure is the **reachability** of the edit map, not missing information. If instead `h*` ghosts like `h_gt`, that is **surprising** and is flagged.

In [ ]:
# [1] Setup: imports, config, load GRU + edits data, teacher-forced bank, linear position probe, helpers.
import sys, os
sys.path.insert(0, "../../..")   # repo root -> import pim
sys.path.insert(0, "../..")      # notebooks/ -> helpers

import numpy as np
import torch
import h5py
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

import pim.eval as eval
from pim.extractors import LinearExtractor, StateDefinition
from pim.editors import probe_decomposition, inject_state, decompose_hidden
from pim.simulator.sim import Scene, SimConfig, frustum_half_width
from pim.simulator.renderer import render_scene
from pim.world_models import load_checkpoint, load_dataset, make_test_loader

torch.manual_seed(0); np.random.seed(0)

CHECKPOINT_PATH = "../../../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
DATA_DIR        = "../../../datasets/4_fixed_refl_inview"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ, N_EDIT, K, W = 2, 64, 15, 10
OUT = "/tmp/counterfactual_history"; os.makedirs(OUT, exist_ok=True)

# Okabe-Ito palette + per-state colors (light theme for metrics; dark theme reserved for waterfalls)
OK = {"blue": "#0072B2", "orange": "#D55E00", "green": "#009E73", "yellow": "#E69F00",
      "pink": "#CC79A7", "sky": "#56B4E9", "grey": "#8a8a8a", "black": "#000000"}
SCOL = {"GT": "#000000", "h* (counterfactual)": OK["blue"], "h*_shared": OK["green"],
        "h_gt (one-frame)": OK["sky"], "h0 (unsteered)": OK["grey"], "h_readout": OK["yellow"]}

model, ckpt_info = load_checkpoint(CHECKPOINT_PATH, device=DEVICE)
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
H = model.hidden_size
ef = edits.edit_frame
N = min(N_EDIT, edits.n_samples)
sim = test.config["dataset"]["sim"]
DT = float(sim["dt"])
print(f"Model : {ckpt_info.run_name} (epoch {ckpt_info.epoch}, val_loss={ckpt_info.val_loss:.5f})")
print(f"Hidden H={H}  device={DEVICE}  edit_frame ef={ef}  N={N}  K={K}  W={W}  dt={DT}")

# --- teacher-forced bank on the test split -> fit the linear position probe (row space = injection subspace) ---
test_loader = make_test_loader(test, batch_size=512, num_workers=6)
_, states_tf = eval.teacher_force(model, test_loader, device=DEVICE)   # (Nt,39,H)
pos_tf = test.positions[:, :-1, :N_OBJ, :]                            # (Nt,39,2,2)
vis_tf = test.is_visible[:, :-1, :N_OBJ].all(axis=2)                  # (Nt,39)
pos_sdef = StateDefinition(name="positions", state_shape=(N_OBJ, 2), extract_fn=lambda b: b["positions"])
lin_pos = LinearExtractor(H, pos_sdef, use_lstsq=True)
lin_pos.fit(states_tf, pos_tf, mask=vis_tf, device=DEVICE)
lin_pos = lin_pos.to(DEVICE).eval()
Ap, bp, Ap_pinv = probe_decomposition(lin_pos)   # Ap:(4,H), bp:(4,), Ap_pinv:(H,4)
print(f"states_tf={states_tf.shape}  position probe fitted (row-space dim = {Ap.shape[0]})")

# --- per-sample render metadata (fixed reflectivities/radii read from the edits HDF5) ---
_hf = h5py.File(edits.h5_path, "r")
REFL = _hf["reflectivities"][:N, :N_OBJ].astype(np.float32)   # (N,2)
RAD  = _hf["radii"][:N, :N_OBJ].astype(np.float32)            # (N,2)
VELS = _hf["velocities"][:N, :, :N_OBJ, :].astype(np.float32) # (N,40,2,2)  (constant velocity)
_hf.close()
COLc = np.tile(np.array([[1, 1, 1]], np.float32), (N_OBJ, 1))
edit_obj = edits.edit_object[:N].astype(int)

def make_cfg(n_frames):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"],
                     x_far=sim["x_far"], n_objects=N_OBJ, radius=sim["radius"], n_frames=n_frames,
                     dt=sim["dt"], obs_res=sim["obs_res"], refl_min=sim["refl_min"], refl_max=sim["refl_max"],
                     fixed_reflectivities=True, obs_noise_std=0.0, boundary="open", always_in_frustum=False)

def render_traj(pos_seq, refl, rad):
    """pos_seq: (T,N_OBJ,2) -> clean obs intensities (T,R) and hit-ids (T,R) via the sim renderer."""
    T = pos_seq.shape[0]
    sc = Scene(positions=pos_seq.astype(np.float32), velocities=np.zeros((T, N_OBJ, 2), np.float32),
               radii=rad, colors=COLc, reflectivities=refl, config=make_cfg(T))
    _, rid, rint = render_scene(sc)
    return rint, rid

@torch.no_grad()
def tf_states(obs_arr, upto):
    """Batched teacher-forcing. obs_arr:(N,T,R). Consume frames 0..upto inclusive -> flat state (N,H)."""
    ot = torch.from_numpy(obs_arr).float().to(DEVICE)
    state = None
    for t in range(upto + 1):
        _, state = model.step(ot[:, t, :], state)
    return model.flat_state(state)   # (N,H)

@torch.no_grad()
def rollout(h_flat_N, n):
    """h_flat_N:(N,H) -> (N,n,R). Step 0 = decode(state); then free-run predict_step."""
    state = model.state_from_flat(h_flat_N)
    obs = [model.decode(state)]
    for _ in range(n - 1):
        p, state = model.predict_step(state); obs.append(p)
    return torch.stack(obs, 1).cpu().numpy()

# QA: the sim renderer reproduces edits.clean_obs exactly (renderer == reconstruct_clean_obs pipeline).
_qa = render_traj(edits.positions[0, :ef + 1, :N_OBJ, :], REFL[0], RAD[0])[0]
print(f"QA render_traj vs edits.clean_obs (frames 0..ef): max|diff| = {np.abs(_qa - edits.clean_obs[0, :ef + 1]).max():.2e}")


---
## 1 — Build the counterfactual history, render its clean observations, check the frustum

For each edit sample: the **edited object** is back-extrapolated `pos_o(t) = target − vel_o·(ef−t)·dt` so it passes through the target at `ef` on its preserved velocity; the **other object** keeps its true history `edits.positions[:, t, other]`. We render clean observations for frames `0..ef` of this world. Because `always_in_frustum=True` during training, the back-extrapolated object may fall **out of the frustum** in early frames (out of distribution) — we report that fraction and also build the **shared-context** variant (real obs for the first `ef−W` frames, counterfactual only for the last `W` frames) as a frustum-robust alternative.

In [ ]:
# [2] Counterfactual + shared-context histories; render clean obs 0..ef; frustum out-of-distribution report.
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32)   # (N,2,2) positions at edit frame (edited obj = target)
cfg1 = make_cfg(1)

# Build per-sample counterfactual position sequences (T = ef+1 frames).
cf_pos   = np.zeros((N, ef + 1, N_OBJ, 2), np.float32)
cf_obs   = np.zeros((N, ef + 1, sim["obs_res"]), np.float32)      # counterfactual clean obs 0..ef
shared_obs = np.zeros((N, ef + 1, sim["obs_res"]), np.float32)    # real 0..ef-W, counterfactual ef-W+1..ef
oof_early = np.zeros(N, bool)      # edited object out-of-frustum in >=1 early frame (0..ef-1)
oof_count = np.zeros(N, int)       # number of early frames out-of-frustum
t_idx = np.arange(ef + 1)
for i in range(N):
    o = edit_obj[i]; other = 1 - o
    vel_o = VELS[i, ef, o]                                        # preserved (constant) velocity
    target = tgt_pos[i, o]
    # edited object: constant-velocity through target at ef
    cf_pos[i, :, o, :] = target[None, :] - vel_o[None, :] * (ef - t_idx)[:, None] * DT
    # other object: true history (its position at ef equals edits.positions as well)
    cf_pos[i, :, other, :] = edits.positions[i, :ef + 1, other, :]
    cf_obs[i] = render_traj(cf_pos[i], REFL[i], RAD[i])[0]
    # shared-context: swap in real clean obs for the early frames
    shared_obs[i] = cf_obs[i].copy()
    shared_obs[i, :ef - W + 1] = edits.clean_obs[i, :ef - W + 1]
    # frustum test on the edited object over the early counterfactual frames 0..ef-1
    x = cf_pos[i, :ef, o, 0]; y = cf_pos[i, :ef, o, 1]; r = float(RAD[i, o])
    in_y = (y + r > sim["y_near"]) & (y - r < sim["y_far"])
    xlim = frustum_half_width(np.clip(y, sim["y_near"], sim["y_far"]), cfg1)
    in_x = np.abs(x) - r < xlim
    inside = in_y & in_x
    oof_count[i] = int((~inside).sum()); oof_early[i] = oof_count[i] > 0

frac_oof = float(oof_early.mean())
print(f"counterfactual obs built: cf_obs={cf_obs.shape}")
print(f"QA cf_obs[:,ef] vs edits.clean_obs[:,ef] (should match — same target world at ef): "
      f"max|diff|={np.abs(cf_obs[:, ef] - edits.clean_obs[:N, ef]).max():.2e}")
print(f"FRUSTUM CAVEAT: {100*frac_oof:.1f}% of samples have the back-extrapolated edited object "
      f"out-of-frustum in >=1 early frame; mean out-of-frustum early frames = {oof_count.mean():.2f} / {ef}")
print("  -> shared-context variant (real first ef-W frames) keeps early history in-distribution.")


In [ ]:
# [3] Build every hidden state; readout sanity + h-space geometry (the reachability argument).
clean = edits.clean_obs[:N].astype(np.float32)                   # realized post-edit clean obs (teleport at ef)
h0    = tf_states(clean,   ef - 1)                               # unsteered: never sees the teleport
h_gt  = tf_states(clean,   ef)                                   # one-frame-evidence (true-state swap)
h_cf  = tf_states(cf_obs,  ef)                                   # counterfactual-history h*
h_sh  = tf_states(shared_obs, ef)                                # shared-context h*
tgt_flat = torch.from_numpy(tgt_pos.reshape(N, N_OBJ * 2)).float().to(DEVICE)
h_ro  = inject_state(h0, tgt_flat, Ap, Ap_pinv, bp)             # readout injection (known-failing)

STATES = {"h* (counterfactual)": h_cf, "h*_shared": h_sh, "h_gt (one-frame)": h_gt,
          "h0 (unsteered)": h0, "h_readout": h_ro}

def readout_rmse(h):  # position-probe readout error vs the target position (RMSE over the 4 coords)
    return float(((h @ Ap.T + bp) - tgt_flat).pow(2).mean().sqrt())
print("position-probe readout RMSE vs target (lower = probe *reads* target position):")
for nm, h in STATES.items():
    print(f"   {nm:22s} {readout_rmse(h):.4f}")

# h-space geometry: distances from h* to the other states, and probe-aligned fraction of (h*-h0).
def dist(a, b): return float((a - b).norm(dim=1).mean())
d_cf_h0  = dist(h_cf, h0); d_cf_hgt = dist(h_cf, h_gt); d_cf_ro = dist(h_cf, h_ro)
h0_norm  = float(h0.norm(dim=1).mean())
delta = h_cf - h0
d_par, d_perp = decompose_hidden(delta, Ap, Ap_pinv)             # row-space (injectable) vs null-space
aligned_frac = float((d_par.norm(dim=1) / delta.norm(dim=1)).mean())
# same fraction for the true-state swap delta, for context
dgt = h_gt - h0
dgt_par, _ = decompose_hidden(dgt, Ap, Ap_pinv)
aligned_frac_gt = float((dgt_par.norm(dim=1) / dgt.norm(dim=1)).mean())
print(f"\n||h*-h0||={d_cf_h0:.3f}   ||h*-h_gt||={d_cf_hgt:.3f}   ||h*-h_readout||={d_cf_ro:.3f}   (mean||h0||={h0_norm:.3f})")
print(f"probe-aligned fraction of (h*-h0)   = {aligned_frac:.3f}  -> only this fraction is reachable by position injection")
print(f"probe-aligned fraction of (h_gt-h0) = {aligned_frac_gt:.3f}  (context)")


---
## 2 — Roll out each state and measure against the sim ground truth

Roll out `K=15` steps from every state and compare to `GT = edits.clean_obs[:, ef:ef+K]`. We form the **ghost ray set** `G` (edited object lit it pre-edit but not at target) and **target ray set** `T` (edited object at target) from single-frame sim renders, then report obs-RMSE-to-GT, ghost ratio, and target-fill — all defined in the table above.

In [ ]:
# [4] Rollouts (K steps) from every state; ghost/target ray masks; GT trajectory + pre-edit context.
ROLL = {nm: rollout(h, K) for nm, h in STATES.items()}            # each (N,K,R)
gt_traj = edits.clean_obs[:N, ef:ef + K, :].astype(np.float32)    # (N,K,R) sim ground truth
N_CTX = 6
ctx_obs = edits.clean_obs[:N, ef - N_CTX:ef, :].astype(np.float32)  # (N,6,R) shared pre-edit context (for waterfalls)
OBS_RES = gt_traj.shape[-1]

# single-frame renders at pre-edit (ef-1) and target (ef) -> ghost / target ray sets
pre_pos = edits.positions[:N, ef - 1, :N_OBJ, :].astype(np.float32)
tgt_id  = np.zeros((N, OBS_RES), np.int64); tgt_int = np.zeros((N, OBS_RES), np.float32)
pre_id  = np.zeros((N, OBS_RES), np.int64)
for i in range(N):
    ti, tid = render_traj(tgt_pos[i][None], REFL[i], RAD[i]); tgt_int[i], tgt_id[i] = ti[0], tid[0]
    _, pid = render_traj(pre_pos[i][None], REFL[i], RAD[i]);   pre_id[i] = pid[0]
ghost_mask  = np.zeros((N, OBS_RES), bool)
target_mask = np.zeros((N, OBS_RES), bool)
for i in range(N):
    ghost_mask[i]  = (pre_id[i] == edit_obj[i]) & (tgt_id[i] != edit_obj[i])
    target_mask[i] = (tgt_id[i] == edit_obj[i])
teleport = np.linalg.norm(tgt_pos - pre_pos, axis=-1)[np.arange(N), edit_obj]
has_ghost = ghost_mask.sum(1) >= 3
SAMPLES = list(np.argsort(teleport * has_ghost)[::-1][:3])       # largest teleport with resolvable ghost
print(f"rollouts: {[k for k in ROLL]}  each {ROLL['h_gt (one-frame)'].shape}")
print(f"ghost rays available: {int(ghost_mask.sum())}   target rays: {int(target_mask.sum())} (over {N}x{OBS_RES})")
print(f"waterfall samples {SAMPLES}  teleport={[round(float(teleport[s]),2) for s in SAMPLES]}")


In [ ]:
# [5] Obs-space metric suite (same metrics/units for every state) -> rendered table. GT column = the floor.
ROLL_ALL = {"GT": gt_traj, **ROLL}
obs_h0 = ROLL["h0 (unsteered)"]
G, T = ghost_mask, target_mask
steps = np.arange(K)

def rmse_gt(o, s):      return float(np.sqrt(((o[:, s, :] - gt_traj[:, s, :]) ** 2).mean()))
def ghost_ratio(o, s):  return float(o[:, s, :][G].mean() / max(obs_h0[:, s, :][G].mean(), 1e-6))
def target_fill(o, s):  return float(o[:, s, :][T].mean() / max(gt_traj[:, s, :][T].mean(), 1e-6))

STEP = {nm: {"rmse":  [rmse_gt(o, s)     for s in steps],
             "ghost": [ghost_ratio(o, s) for s in steps],
             "fill":  [target_fill(o, s) for s in steps]} for nm, o in ROLL_ALL.items()}

def md_table(rows, cols, row_hdr=""):
    lines = ["| " + row_hdr + " | " + " | ".join(d for _, d, _ in cols) + " |",
             "|" + "---|" * (len(cols) + 1)]
    for rn, v in rows.items():
        cells = [("nan" if isinstance(v[k], float) and np.isnan(v[k]) else f.format(v[k])) for k, _, f in cols]
        lines.append("| **" + rn + "** | " + " | ".join(cells) + " |")
    return Markdown("\n".join(lines))

rows = {}
for nm in ROLL_ALL:
    s = STEP[nm]
    rows[nm] = dict(rmse0=s["rmse"][0], rmse_mean=float(np.mean(s["rmse"])), rmse_end=s["rmse"][-1],
                    ghost0=s["ghost"][0], ghost_end=s["ghost"][-1],
                    fill0=s["fill"][0], fill_end=s["fill"][-1])
COLS = [("rmse0", "RMSE→GT s0 ↓", "{:.4f}"), ("rmse_mean", "RMSE→GT mean ↓", "{:.4f}"),
        ("rmse_end", "RMSE→GT s14 ↓", "{:.4f}"), ("ghost0", "ghost s0 ↓", "{:.3f}"),
        ("ghost_end", "ghost s14 ↓", "{:.3f}"), ("fill0", "target-fill s0 →1", "{:.3f}"),
        ("fill_end", "target-fill s14 →1", "{:.3f}")]
print("Obs-space metrics vs sim GT (RMSE units = intensity in [0,1]; ghost 0=clean; fill 1=object at target like GT):")
display(md_table(rows, COLS, row_hdr="state"))

# machine-readable headline check
r_cf, r_gt = rows["h* (counterfactual)"]["rmse_mean"], rows["h_gt (one-frame)"]["rmse_mean"]
print(f"\nHEADLINE  h* mean RMSE→GT={r_cf:.4f}  vs  h_gt={r_gt:.4f}  (ratio h*/h_gt = {r_cf/max(r_gt,1e-9):.2f})")
print(f"          h* ghost s0={rows['h* (counterfactual)']['ghost0']:.3f}  h_gt ghost s0={rows['h_gt (one-frame)']['ghost0']:.3f}"
      f"  |  h* fill s0={rows['h* (counterfactual)']['fill0']:.3f}  h_gt fill s0={rows['h_gt (one-frame)']['fill0']:.3f}")


In [ ]:
# [6] Bootstrap 95% CI on the per-sample mean obs-RMSE→GT for each state (N=64 is small — quantify spread).
def per_sample_mean_rmse(o):        # (N,) mean over steps+rays of RMSE to GT, per sample
    return np.sqrt(((o - gt_traj) ** 2).mean(axis=(1, 2)))
rng = np.random.RandomState(0); B = 2000
boot_rows = {}
for nm, o in ROLL_ALL.items():
    ps = per_sample_mean_rmse(o)
    idx = rng.randint(0, N, size=(B, N))
    means = ps[idx].mean(1)
    boot_rows[nm] = dict(mean=float(ps.mean()), lo=float(np.percentile(means, 2.5)), hi=float(np.percentile(means, 97.5)))
display(md_table(boot_rows,
                 [("mean", "mean RMSE→GT ↓", "{:.4f}"), ("lo", "95% CI lo", "{:.4f}"), ("hi", "95% CI hi", "{:.4f}")],
                 row_hdr="state"))
# paired bootstrap on the h*-vs-h_gt gap (same resampled samples)
psi_cf = per_sample_mean_rmse(ROLL["h* (counterfactual)"]); psi_gt = per_sample_mean_rmse(ROLL["h_gt (one-frame)"])
idx = rng.randint(0, N, size=(B, N)); gap = (psi_gt[idx] - psi_cf[idx]).mean(1)
gap_mean, gap_lo, gap_hi = float(gap.mean()), float(np.percentile(gap, 2.5)), float(np.percentile(gap, 97.5))
print(f"paired gap (h_gt - h*) mean RMSE→GT = {gap_mean:.4f}  95% CI [{gap_lo:.4f}, {gap_hi:.4f}]"
      f"  (>0 ⇒ h* strictly cleaner than the one-frame state)")


---
## 3 — Figures

In [ ]:
# [7] Fig 1 — per-step curves vs sim GT: (a) obs-RMSE→GT, (b) ghost ratio, (c) target-fill. Light academic theme.
plt.style.use("default")
order = ["GT", "h* (counterfactual)", "h*_shared", "h_gt (one-frame)", "h0 (unsteered)", "h_readout"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
panels = [("rmse", "obs-RMSE → GT (↓)", "(a) does it match the sim trajectory?", None),
          ("ghost", "ghost ratio (↓, 0=clean)", "(b) is the ghost gone?", 0.0),
          ("fill", "target-fill (→1)", "(c) is the object at target?", 1.0)]
for ax, (key, ylab, ttl, ref) in zip(axes, panels):
    for nm in order:
        lw = 2.4 if nm == "h* (counterfactual)" else (1.6 if nm == "GT" else 1.4)
        ls = "--" if nm == "GT" else "-"
        ax.plot(steps, STEP[nm][key], color=SCOL[nm], ls=ls, lw=lw,
                marker="o" if nm == "h* (counterfactual)" else None, ms=3, label=nm)
    if ref is not None:
        ax.axhline(ref, color="0.6", ls=":", lw=1)
    ax.set_xlabel("rollout step s  (sim frame ef+s)"); ax.set_ylabel(ylab); ax.set_title(ttl)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=7, ncol=1)
fig.suptitle("Fig 1 — per-step observation-space metrics vs sim ground truth (GT = clean_obs[ef:ef+K])",
             y=1.02, fontsize=13)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_per_step.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); print("saved fig1_per_step.png")


In [ ]:
# [8] Fig 2 — waterfalls (dark, world_model_eval style): GT | h* | h*_shared | h_gt | h0 | h_readout.
#      Green = where the edited object SHOULD be (target); red dashed = pre-edit/ghost location.
def centroid(mask_row):
    idx = np.where(mask_row)[0]; return idx.mean() if idx.size else np.nan
wf_order = ["GT", "h* (counterfactual)", "h*_shared", "h_gt (one-frame)", "h0 (unsteered)", "h_readout"]
plt.style.use("dark_background")
fig, axes = plt.subplots(len(SAMPLES), len(wf_order),
                         figsize=(2.5 * len(wf_order), 3.1 * len(SAMPLES)), squeeze=False)
for r, smp in enumerate(SAMPLES):
    tgt_cx = centroid(tgt_id[smp] == edit_obj[smp]); pre_cx = centroid(pre_id[smp] == edit_obj[smp])
    for c, nm in enumerate(wf_order):
        ax = axes[r][c]
        strip = ROLL_ALL[nm][smp]                                # (K,R)
        img = np.vstack([ctx_obs[smp], strip])                   # prepend shared pre-edit context
        ax.imshow(img, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.axhline(N_CTX - 0.5, color="#00E676", lw=0.8, alpha=0.5)   # edit-frame boundary
        if not np.isnan(tgt_cx): ax.axvline(tgt_cx, color="#00E676", lw=1.4)
        if not np.isnan(pre_cx): ax.axvline(pre_cx, color="#FF5252", ls="--", lw=1.4)
        if r == 0: ax.set_title(nm, fontsize=8.5)
        if c == 0: ax.set_ylabel(f"smp {smp}\n(teleport {teleport[smp]:.1f})\nframe", fontsize=8)
        ax.set_xlabel("ray", fontsize=8); ax.tick_params(labelsize=7)
fig.suptitle("Fig 2 — waterfalls: rows = samples, columns = states. Green = target, red dashed = ghost. "
             "First 6 rows are shared pre-edit context.", y=1.005, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_waterfalls.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); plt.style.use("default"); print("saved fig2_waterfalls.png")


In [ ]:
# [9] Fig 3 — h-space geometry (the reachability argument): (a) distances from h*, (b) probe-aligned fraction.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
ax = axes[0]
dnames = ["‖h*−h0‖", "‖h*−h_gt‖", "‖h*−h_readout‖"]
dvals  = [d_cf_h0, d_cf_hgt, d_cf_ro]
ax.bar(dnames, dvals, color=[OK["grey"], OK["sky"], OK["yellow"]])
ax.axhline(h0_norm, color="k", ls=":", lw=1, label=f"mean ‖h0‖ = {h0_norm:.2f}")
ax.set_ylabel("mean L2 distance in h-space"); ax.set_title("(a) how far is h* from the reachable states?")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
for i, v in enumerate(dvals):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)

ax = axes[1]
fnames = ["h*−h0", "h_gt−h0"]
fvals  = [aligned_frac, aligned_frac_gt]
ax.bar(fnames, fvals, color=[OK["blue"], OK["sky"]])
ax.axhline(1.0, color="0.6", ls=":", lw=1)
ax.set_ylim(0, 1.02); ax.set_ylabel("probe-aligned fraction  ‖P_row(A)·Δ‖ / ‖Δ‖")
ax.set_title("(b) fraction of the edit reachable by position injection")
ax.grid(alpha=0.3, axis="y")
for i, v in enumerate(fvals):
    ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
fig.suptitle("Fig 3 — h-space geometry: h* is far from h0 and mostly OUTSIDE the position-probe row space "
             "(→ unreachable by a low-dim injection)", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_h_geometry.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig); print("saved fig3_h_geometry.png")


In [ ]:
# [10] Control — the model's own free-running rollout FLOOR on normal (un-edited) in-distribution trajectories.
#      This is the best obs-RMSE any teacher-forced state can achieve when rolled out K steps: the model is
#      not a perfect predictor, so h* cannot beat this. It tells us whether h*'s residual is "model floor" or a deficiency.
n_ctrl = 256
test_clean = test.clean_obs[:n_ctrl].astype(np.float32)              # (n,40,R) normal trajectories, no edit
h_ctrl = tf_states(test_clean, ef - 1)                              # consume 0..ef-1 -> decode predicts ef (aligned like h0)
roll_ctrl = rollout(h_ctrl, K)
gt_ctrl = test_clean[:, ef:ef + K, :]
freerun_floor = float(np.sqrt(((roll_ctrl - gt_ctrl) ** 2).mean()))
freerun_floor0 = float(np.sqrt(((roll_ctrl[:, 0] - gt_ctrl[:, 0]) ** 2).mean()))
print(f"model free-running rollout FLOOR (normal trajectories, n={n_ctrl}): "
      f"mean obs-RMSE={freerun_floor:.4f}   step-0 obs-RMSE={freerun_floor0:.4f}")
print(f"  -> h* mean RMSE→GT={rows['h* (counterfactual)']['rmse_mean']:.4f} sits {rows['h* (counterfactual)']['rmse_mean']-freerun_floor:+.4f} "
      f"above this floor; h_gt={rows['h_gt (one-frame)']['rmse_mean']:.4f} sits {rows['h_gt (one-frame)']['rmse_mean']-freerun_floor:+.4f} above.")
print(f"  GT-floor ghost (geometric, objects drift into the ray set): s0={rows['GT']['ghost0']:.3f} s14={rows['GT']['ghost_end']:.3f}"
      f"   -> h* ghost s0 excess above GT floor = {rows['h* (counterfactual)']['ghost0']-rows['GT']['ghost0']:+.3f}")


---
## 4 — Verdict (decision rule applied programmatically)

Two references bound the metrics and make the verdict honest:
- **Model free-running floor** (cell `[10]`): the best obs-RMSE any teacher-forced state can reach when rolled out `K` steps — the GRU is not a perfect predictor, so `h*` cannot beat this.
- **GT-floor ghost**: even the true post-edit `clean_obs` has a nonzero ghost ratio (objects drift into the fixed ghost ray set), so "ghost ≈ 0" means "≈ the GT floor", not literally 0.

Cell `[11]` applies a three-way rule: **CLEAN PASS** (`h*` beats `h_gt` on every metric, places the object at target, and its RMSE sits at the model free-running floor with ghost ≈ GT floor), **QUALIFIED PASS** (`h*` decisively beats `h_gt` and places the object at target, but keeps a residual ghost above the GT floor), or **SURPRISE** (`h*` no cleaner than the one-frame state — flag loudly).

In [ ]:
# [11] Apply the three-way decision rule; print the verdict + the numbers that decide it.
cf, gt, gtf = rows["h* (counterfactual)"], rows["h_gt (one-frame)"], rows["GT"]

# (1) Does h* decisively beat the one-frame-evidence state, and place the object at target?
beats_onefrmame = (gap_lo > 0) and (cf["ghost0"] < 0.6 * gt["ghost0"]) and (cf["fill0"] > gt["fill0"] + 0.2) \
                  and (cf["fill0"] > 0.8)
# (2) Is it a *crisp* clean render — RMSE at the model free-running floor, ghost at the GT floor?
at_floor = (cf["rmse_mean"] <= freerun_floor + 0.03) and (cf["ghost0"] - gtf["ghost0"] < 0.08)

if beats_onefrmame and at_floor:
    verdict = "CLEAN PASS — a ghost-free clean-edit state EXISTS (renders at the model floor)"
elif beats_onefrmame:
    verdict = "QUALIFIED PASS — h* is DECISIVELY cleaner than the one-frame state and places the object at target, "\
              "but keeps a residual ghost above the GT floor"
else:
    verdict = "SURPRISE — h* is NOT meaningfully cleaner than the one-frame state; FLAG loudly"

print("=" * 92)
print("VERDICT:", verdict)
print("=" * 92)
print(f"  h*  mean RMSE→GT = {cf['rmse_mean']:.4f}  (h_gt {gt['rmse_mean']:.4f}, model free-run floor {freerun_floor:.4f}) "
      f"-> h* is {gt['rmse_mean']/max(cf['rmse_mean'],1e-9):.2f}x cleaner than h_gt; {cf['rmse_mean']-freerun_floor:+.4f} above floor")
print(f"  paired bootstrap gap (h_gt - h*) = {gap_mean:.4f}  95% CI [{gap_lo:.4f}, {gap_hi:.4f}]  (robustly > 0)")
print(f"  h*  ghost s0/s14 = {cf['ghost0']:.3f}/{cf['ghost_end']:.3f}  (h_gt {gt['ghost0']:.3f}/{gt['ghost_end']:.3f}; "
      f"GT floor {gtf['ghost0']:.3f}/{gtf['ghost_end']:.3f})  -> h* ghost excess above GT floor = {cf['ghost0']-gtf['ghost0']:+.3f} (s0)")
print(f"  h*  fill  s0/s14 = {cf['fill0']:.3f}/{cf['fill_end']:.3f}  (h_gt {gt['fill0']:.3f}/{gt['fill_end']:.3f}; GT 1.000) "
      f"-> object IS at target (fill≈1), unlike the one-frame state")
print(f"  shared-context h*: RMSE {rows['h*_shared']['rmse_mean']:.4f}  ghost s0 {rows['h*_shared']['ghost0']:.3f}  "
      f"fill s0 {rows['h*_shared']['fill0']:.3f}  (frustum-robust — same conclusion)")
print(f"  REACHABILITY: ||h*-h0|| = {d_cf_h0:.2f} ≈ mean||h0|| {h0_norm:.2f}; only {100*aligned_frac:.1f}% of (h*-h0) "
      f"lies in the 4-D position-probe row space")
print(f"                -> a pseudoinverse position injection cannot reach h* (readout-injection RMSE→GT = {rows['h_readout']['rmse_mean']:.4f}, "
      f"no better than unsteered {rows['h0 (unsteered)']['rmse_mean']:.4f})")
print(f"  frustum caveat: {100*frac_oof:.1f}% of samples out-of-frustum in >=1 early counterfactual frame "
      f"(shared-context variant confirms the caveat does not change the verdict).")
